<a href="https://colab.research.google.com/github/anushah-200/SATARK_AI/blob/main/notebooks/02_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
PROJECT_PATH = "/content/drive/MyDrive/SATARK_AI"

In [9]:
RAW_PATH = f"{PROJECT_PATH}/data/raw"
PROCESSED_PATH = f"{PROJECT_PATH}/data/processed"
SYNTHETIC_PATH = f"{PROJECT_PATH}/data/synthetic"
MODEL_PATH = f"{PROJECT_PATH}/models"
OUTPUT_PATH = f"{PROJECT_PATH}/outputs"

In [10]:

import os

print(os.listdir(PROJECT_PATH))

['data', 'outputs', 'src', 'notebooks', 'models']


In [11]:
import pandas as pd

file_path = f"{RAW_PATH}/meteostat_data.csv"

meteostat_data = pd.read_csv(file_path)

print("Shape:", meteostat_data.shape)
meteostat_data.head()

Shape: (43100, 10)


,timestamp,station_id,station_name,latitude,longitude,temperature,humidity,pressure,wind_speed,wind_direction
0,2025-01-01 00:00:00,42131,Hissar,29.1667,75.7333,5.4,97.0,1020.7,0.0,0.0
1,2025-01-01 01:00:00,42131,Hissar,29.1667,75.7333,6.1,98.0,1019.5,5.0,283.0
2,2025-01-01 02:00:00,42131,Hissar,29.1667,75.7333,6.1,94.0,1019.8,5.4,274.0
3,2025-01-01 03:00:00,42131,Hissar,29.1667,75.7333,8.0,97.0,1021.5,0.0,0.0
4,2025-01-01 04:00:00,42131,Hissar,29.1667,75.7333,7.6,96.0,1020.8,6.8,276.0


In [12]:
import os

print("Project exists:", os.path.exists(PROJECT_PATH))
print("Raw data exists:", os.path.exists(f"{RAW_PATH}/meteostat_data.csv"))

print("\nRaw files:")
print(os.listdir(RAW_PATH))

Project exists: True
Raw data exists: True

Raw files:
['meteostat_data.csv']


In [13]:
file_path = f"{RAW_PATH}/meteostat_data.csv"

meteostat_data = pd.read_csv(file_path)

print("Dataset shape:", meteostat_data.shape)
print("Columns:", list(meteostat_data.columns))

Dataset shape: (43100, 10)
Columns: ['timestamp', 'station_id', 'station_name', 'latitude', 'longitude', 'temperature', 'humidity', 'pressure', 'wind_speed', 'wind_direction']


In [14]:
meteostat_data.head()

,timestamp,station_id,station_name,latitude,longitude,temperature,humidity,pressure,wind_speed,wind_direction
0,2025-01-01 00:00:00,42131,Hissar,29.1667,75.7333,5.4,97.0,1020.7,0.0,0.0
1,2025-01-01 01:00:00,42131,Hissar,29.1667,75.7333,6.1,98.0,1019.5,5.0,283.0
2,2025-01-01 02:00:00,42131,Hissar,29.1667,75.7333,6.1,94.0,1019.8,5.4,274.0
3,2025-01-01 03:00:00,42131,Hissar,29.1667,75.7333,8.0,97.0,1021.5,0.0,0.0
4,2025-01-01 04:00:00,42131,Hissar,29.1667,75.7333,7.6,96.0,1020.8,6.8,276.0


In [15]:
meteostat_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43100 entries, 0 to 43099
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   timestamp       43100 non-null  object 
 1   station_id      43100 non-null  int64  
 2   station_name    43100 non-null  object 
 3   latitude        43100 non-null  float64
 4   longitude       43100 non-null  float64
 5   temperature     43100 non-null  float64
 6   humidity        43100 non-null  float64
 7   pressure        43100 non-null  float64
 8   wind_speed      43100 non-null  float64
 9   wind_direction  42950 non-null  float64
dtypes: float64(7), int64(1), object(2)
memory usage: 3.3+ MB


In [16]:
meteostat_data.isnull().sum()

,0
timestamp,0
station_id,0
station_name,0
latitude,0
longitude,0
temperature,0
humidity,0
pressure,0
wind_speed,0
wind_direction,150


In [17]:
meteostat_data["timestamp"] = pd.to_datetime(
    meteostat_data["timestamp"],
    errors="coerce"
)

print(meteostat_data["timestamp"].dtype)
print("Invalid timestamps:", meteostat_data["timestamp"].isna().sum())

datetime64[ns]
Invalid timestamps: 0


In [18]:
duplicates = meteostat_data.duplicated(
    subset=["station_id", "timestamp"]
).sum()

print("Duplicate station-timestamp records:", duplicates)

Duplicate station-timestamp records: 0


In [19]:
meteostat_data = meteostat_data.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

In [20]:
core_variables = [
    "temperature",
    "pressure",
    "humidity"
]

print(meteostat_data[core_variables].describe())

        temperature      pressure      humidity
count  43100.000000  43100.000000  43100.000000
mean      24.361745   1008.333842     67.884200
std        7.864331      7.234360     21.409544
min        4.000000    991.500000      5.000000
25%       18.300000   1002.100000     53.000000
50%       26.000000   1008.200000     71.000000
75%       30.100000   1014.800000     85.000000
max       44.000000   1026.300000    100.000000


In [21]:
print(meteostat_data.isnull().sum())

timestamp           0
station_id          0
station_name        0
latitude            0
longitude           0
temperature         0
humidity            0
pressure            0
wind_speed          0
wind_direction    150
dtype: int64


In [22]:
print(
    meteostat_data[
        ["temperature", "pressure", "humidity"]
    ].isnull().sum()
)

temperature    0
pressure       0
humidity       0
dtype: int64


In [23]:
meteostat_data["hour"] = meteostat_data["timestamp"].dt.hour
meteostat_data["day"] = meteostat_data["timestamp"].dt.day
meteostat_data["month"] = meteostat_data["timestamp"].dt.month
meteostat_data["day_of_week"] = meteostat_data["timestamp"].dt.dayofweek
meteostat_data["day_of_year"] = meteostat_data["timestamp"].dt.dayofyear

In [24]:
meteostat_data[
    [
        "timestamp",
        "hour",
        "day",
        "month",
        "day_of_week",
        "day_of_year"
    ]
].head(10)

,timestamp,hour,day,month,day_of_week,day_of_year
0,2025-01-01 00:00:00,0,1,1,2,1
1,2025-01-01 01:00:00,1,1,1,2,1
2,2025-01-01 02:00:00,2,1,1,2,1
3,2025-01-01 03:00:00,3,1,1,2,1
4,2025-01-01 04:00:00,4,1,1,2,1
5,2025-01-01 05:00:00,5,1,1,2,1
6,2025-01-01 06:00:00,6,1,1,2,1
7,2025-01-01 07:00:00,7,1,1,2,1
8,2025-01-01 08:00:00,8,1,1,2,1
9,2025-01-01 09:00:00,9,1,1,2,1


In [25]:
def get_time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

meteostat_data["time_period"] = meteostat_data["hour"].apply(
    get_time_period
)

In [26]:
meteostat_data["time_period"].value_counts()

,count
time_period,
Night,14360
Morning,12570
Afternoon,8984
Evening,7186


In [27]:
def create_rolling_features(df, column):

    rolling_mean = (
        df.groupby("station_id", group_keys=False)
          .apply(
              lambda x: x.set_index("timestamp")[column]
                        .shift(1)
                        .rolling("24h", min_periods=3)
                        .mean()
          )
          .reset_index(level=0, drop=True)
    )

    rolling_std = (
        df.groupby("station_id", group_keys=False)
          .apply(
              lambda x: x.set_index("timestamp")[column]
                        .shift(1)
                        .rolling("24h", min_periods=3)
                        .std()
          )
          .reset_index(level=0, drop=True)
    )

    df[f"{column}_rolling_mean_24h"] = rolling_mean.values
    df[f"{column}_rolling_std_24h"] = rolling_std.values

    df[f"{column}_deviation_24h"] = (
        df[column] -
        df[f"{column}_rolling_mean_24h"]
    )

    std = df[f"{column}_rolling_std_24h"].replace(0, np.nan)

    df[f"{column}_zscore_24h"] = (
        df[f"{column}_deviation_24h"] / std
    )

    return df

In [28]:
for column in ["temperature", "pressure", "humidity"]:
    meteostat_data = create_rolling_features(
        meteostat_data,
        column
    )

/tmp/ipykernel_757/2547466137.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_757/2547466137.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_757/2547466137.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operat

In [29]:
print(meteostat_data.columns.tolist())

['timestamp', 'station_id', 'station_name', 'latitude', 'longitude', 'temperature', 'humidity', 'pressure', 'wind_speed', 'wind_direction', 'hour', 'day', 'month', 'day_of_week', 'day_of_year', 'time_period', 'temperature_rolling_mean_24h', 'temperature_rolling_std_24h', 'temperature_deviation_24h', 'temperature_zscore_24h', 'pressure_rolling_mean_24h', 'pressure_rolling_std_24h', 'pressure_deviation_24h', 'pressure_zscore_24h', 'humidity_rolling_mean_24h', 'humidity_rolling_std_24h', 'humidity_deviation_24h', 'humidity_zscore_24h']


In [30]:
safdarjung = meteostat_data[
    meteostat_data["station_id"] == "42182"
].copy()

safdarjung[
    [
        "timestamp",
        "temperature",
        "temperature_rolling_mean_24h",
        "temperature_deviation_24h",
        "temperature_zscore_24h"
    ]
].head(30)

,timestamp,temperature,temperature_rolling_mean_24h,temperature_deviation_24h,temperature_zscore_24h


In [31]:
processed_file = (
    f"{PROCESSED_PATH}/processed_meteostat_data.csv"
)

meteostat_data.to_csv(
    processed_file,
    index=False
)

print("Saved processed dataset to:")
print(processed_file)

Saved processed dataset to:
/content/drive/MyDrive/SATARK_AI/data/processed/processed_meteostat_data.csv


In [32]:
processed_file = (
    f"{PROCESSED_PATH}/processed_meteostat_data.csv"
)

meteostat_data.to_csv(
    processed_file,
    index=False
)

print("Saved processed dataset to:")
print(processed_file)

Saved processed dataset to:
/content/drive/MyDrive/SATARK_AI/data/processed/processed_meteostat_data.csv
